# April 05
我把utils.py里面的Reward Signals给重新整理了下然后现在有6种不同的signals.
直接用train_ray_selfplay.py 去从头训练这个新的Reward shape

Submit 了2个Train 一个CPU版本一个GPU版本
等看看训练结果，如果GPU没问题以后就只用GPU训练就好了。

NEXT:
出结果后和ceia_baseline_agent 打个比赛看看效果。

In [ ]:
#SBATCH --job-name=GPU_S2_selfplay
#SBATCH -N1 --ntasks-per-node=12 --gres=gpu:V100:1
#SBATCH --mem-per-cpu=6G
#SBATCH -t12:00:00
#SBATCH --output=soccerstwos-%j.out
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=frank.yang@gatech.edu


source ~/miniconda3/etc/profile.d/conda.sh
conda activate soccertwos

# Fix protobuf version conflict with ray==1.4.0
export PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python
pip install "numpy==1.23.5" -q  # fix numpy version conflict with ray==1.4.0

# Run the training script
/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/bin/python train_ray_selfplay.py


# April 06
GPU在这个项目里只是把 PPO 的模型更新（forward/backward pass）放到 GPU 上，但 soccer-twos 的环境是 Unity 进程，仿真本身无法用 GPU 加速，而且 MLP 太小，GPU 收益可以忽略不计。
之后训练直接用CPU的就行。


### 当前脚本不支持多节点
train_ray_selfplay.py:40
ray.init()  # 只初始化本地 Ray，不知道其他节点
多节点需要改成 ray.init(address='auto') 并在 Slurm 脚本里手动启动 Ray 集群（head + workers），比较复杂。

### 修复不能继续训练的问题
问题：checkpoint 序列化时的类型信息丢失。
具体链路是这样的：
Ray 保存 checkpoint 时，把 optimizer state dict（里面是 torch tensor）先转成 numpy array，再用 pickle 序列化存盘
某些 tensor（比如 Adam 的 step 计数器，是一个标量 int64）在转换过程中被存成了 numpy.object_ 类型，而不是具体的数值类型
恢复时 Ray 想把这些 numpy array 转回 torch tensor，调用 torch.from_numpy()，但 torch 拒绝转换 object_ 类型

修复方式：
去到/storage/ice1/7/4/fyang365/.conda/conda_envs/soccertwos/lib/python3.8/site-packages/ray/rllib/utils/torch_utils.py
把def mapping(item)里面的
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    tensor = torch.from_numpy(np.asarray(item))
```
替换为：
```
# Everything else: Convert to numpy, then wrap as torch tensor.
else:
    arr = np.asarray(item)
    if arr.dtype == object:
        arr = np.array(arr.tolist(), dtype=np.float32)
    # Scalar values (like lr) should stay as Python scalars, not tensors
    if arr.ndim == 0:
        return arr.item()
    tensor = torch.from_numpy(arr)
```


# April 07
用Ray 1.4版本重新跑

# April 08
用`python -m soccer_twos.evaluate     -m1 reward_shaping_ppo_agent     -m2 ceia_baseline_agent     -e 100`
比较了下 Ray1.4 的reward shaping 版本和 ceia 发现：
reward shaping 版本比 TA 的原始 ceia baseline 强 9 个百分点。
需要继续训练

# April 09
目前有几个训练的结果
- `PPO_Soccer_4923f_00000_0_2026-04-08_21-56-12` 这个是Ray1.13 的1300版本
- `PPO_Soccer_36ca0_00000_0_2026-04-07_18-00-41` 这个是Ray1.4 的1000 版本
- `PPO_Soccer_1874b_00000_0_2026-04-09_13-18-16` Ray1.4版本 所有的环境记录在 environment.yml 里面



# April 10
`PPO_Soccer_bc696_00000_0_2026-04-09_14-12-58` 
- Ray1.4 并且消除了 `DeprecationWarning: wrapping <function policy_mapping_fn at `  
- 训练出`scratch/soccer-twos/ray_results/PPO_selfplay_rec/PPO_Soccer_bc696_00000_0_2026-04-09_14-12-58/checkpoint_000700/checkpoint-700`
- 从这儿继续训练 `PPO_Soccer_0b340_00000_0_2026-04-10_15-47-02`

### 对比测试：
`python -m soccer_twos.watch -m1 reward_shaping_ppo_agent -m2 ceia_baseline_agent`
- 上面这个需要显示出来只能本地
- 服务器上用这个：
```
python -m soccer_twos.evaluate \
    -m1 reward_shaping_ppo_agent \
    -m2 ceia_baseline_agent \
    -e 100
```

### checkpoint-700 拉完了
```
Progress: |██████████████████████████████████████████████████| 100 / 100 episodes completed
episode_len_mean: 79.32
episode_reward_max: -0.013599991798400879
episode_reward_mean: -0.15827199816703796
episode_reward_min: -1.6103999614715576
episodes_this_eval: 100
policies:
  ceia_baseline_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 19
      policy_blue_team_reward_max: 1.981600046157837
      policy_blue_team_reward_mean: 0.39344000816345215
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.62
      policy_blue_team_wins: 31
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 27
      policy_orange_team_reward_max: 1.9847999811172485
      policy_orange_team_reward_mean: -0.2342880219221115
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.46
      policy_orange_team_wins: 23
    policy_draws: 0
    policy_losses: 46
    policy_reward_max: 1.9847999811172485
    policy_reward_mean: 0.07957600802183151
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.54
    policy_wins: 54
  reward_shaping_ppo_agent:
    blue_team:
      policy_blue_team_draws: 0
      policy_blue_team_losses: 23
      policy_blue_team_reward_max: 1.979599952697754
      policy_blue_team_reward_mean: 0.06441599130630493
      policy_blue_team_reward_min: -2.0
      policy_blue_team_total_games: 50
      policy_blue_team_win_rate: 0.54
      policy_blue_team_wins: 27
    orange_team:
      policy_orange_team_draws: 0
      policy_orange_team_losses: 31
      policy_orange_team_reward_max: 1.9864000082015991
      policy_orange_team_reward_mean: -0.5401120185852051
      policy_orange_team_reward_min: -2.0
      policy_orange_team_total_games: 50
      policy_orange_team_win_rate: 0.38
      policy_orange_team_wins: 19
    policy_draws: 0
    policy_losses: 54
    policy_reward_max: 1.9864000082015991
    policy_reward_mean: -0.23784799873828888
    policy_reward_min: -2.0
    policy_total_games: 100
    policy_win_rate: 0.46
    policy_wins: 46
```